# EMNIST Character Classifier — Training Notebook

**Stage 1 OCR | CharClassifier CNN**

This notebook trains a CNN on the EMNIST `byclass` dataset (62 classes: digits 0–9, A–Z, a–z) with Gaussian and salt-and-pepper noise augmentation.

**Steps:**
1. Install & import dependencies
2. Define noise transforms
3. Load EMNIST dataset
4. Visualise sample images
5. Build the model
6. Train with live loss tracking
7. Plot training curves
8. Evaluate per-noise accuracy (grad requirement)
9. Save model weights

## 1. Install Dependencies

In [ ]:
%pip install torch torchvision numpy matplotlib --quiet

## 2. Imports & Config

In [ ]:
import os, sys, math
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Point to project root so we can import ocr_service.model
sys.path.insert(0, os.path.abspath(".."))
from ocr_service.model import CharClassifier

# ── Config — change these as needed ──────────────────────────────────────────
EPOCHS      = 20
BATCH_SIZE  = 128
LR          = 1e-3
NUM_CLASSES = 62
DATA_DIR    = os.path.abspath("../data")
MODEL_DIR   = os.path.abspath("../models")
SAVE_PATH   = os.path.join(MODEL_DIR, "char_classifier.pth")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

DEVICE = ("cuda"  if torch.cuda.is_available() else
          "mps"   if torch.backends.mps.is_available() else
          "cpu")
print(f"Device : {DEVICE}")
print(f"Epochs : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  LR : {LR}")

## 3. Noise Transforms

In [ ]:
class AddGaussianNoise:
    """Additive Gaussian noise, sigma ~ Uniform(lo, hi)."""
    def __init__(self, lo=0.05, hi=0.15):
        self.lo, self.hi = lo, hi

    def __call__(self, t):
        sigma = np.random.uniform(self.lo, self.hi)
        return (t + sigma * torch.randn_like(t)).clamp(0., 1.)


class AddSaltAndPepperNoise:
    """Random pixels set to 0 or 1, density ~ Uniform(lo, hi)."""
    def __init__(self, lo=0.02, hi=0.08):
        self.lo, self.hi = lo, hi

    def __call__(self, t):
        density = np.random.uniform(self.lo, self.hi)
        mask = torch.rand_like(t)
        out  = t.clone()
        out[mask < density / 2]       = 0.   # pepper
        out[mask > 1 - density / 2]   = 1.   # salt
        return out


class RandomNoise:
    """50% chance of applying Gaussian or salt-and-pepper, each equally likely."""
    def __init__(self, p=0.5):
        self.p = p
        self.g  = AddGaussianNoise()
        self.sp = AddSaltAndPepperNoise()

    def __call__(self, t):
        if np.random.rand() > self.p:
            return t
        return self.g(t) if np.random.rand() < 0.5 else self.sp(t)

print("Noise transforms defined.")

## 4. Load EMNIST Dataset

In [ ]:
base_tf  = [T.Resize((32, 32)), T.ToTensor()]
train_tf = T.Compose(base_tf + [RandomNoise(p=0.5)])
val_tf   = T.Compose(base_tf)

print("Downloading EMNIST byclass (may take a few minutes on first run)...")
train_ds = torchvision.datasets.EMNIST(root=DATA_DIR, split="byclass",
                                        train=True,  download=True, transform=train_tf)
val_ds   = torchvision.datasets.EMNIST(root=DATA_DIR, split="byclass",
                                        train=False, download=True, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Classes       : {NUM_CLASSES}")

## 5. Visualise Sample Images

In [ ]:
# EMNIST byclass label map: 0-9 digits, 10-35 uppercase, 36-61 lowercase
label_chars = (
    [str(i) for i in range(10)] +
    [chr(c) for c in range(ord('A'), ord('Z')+1)] +
    [chr(c) for c in range(ord('a'), ord('z')+1)]
)

fig, axes = plt.subplots(3, 10, figsize=(14, 5))
fig.suptitle("EMNIST samples (with noise augmentation)", fontsize=13)
for ax, (img, lbl) in zip(axes.flat, train_ds):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label_chars[lbl], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Build Model

In [ ]:
model     = CharClassifier(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5, verbose=True
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model      : CharClassifier")
print(f"Parameters : {total_params:,}")
print(f"Optimizer  : Adam  (lr={LR})")
print(model)

## 7. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(va_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), SAVE_PATH)
        saved = " ✓ saved"
    else:
        saved = ""

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}{saved}")

print(f"\nBest val accuracy : {best_val_acc:.4f}")
if best_val_acc < 0.95:
    print("WARNING: below 95% target — consider more epochs or lower LR.")

## 8. Plot Training Curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_range, history["train_loss"], label="Train loss")
ax1.plot(epochs_range, history["val_loss"],   label="Val loss")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history["train_acc"], label="Train acc")
ax2.plot(epochs_range, history["val_acc"],   label="Val acc")
ax2.axhline(0.95, color="red", linestyle="--", label="95% target")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)

plt.suptitle("CharClassifier — Training History", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Per-Noise Accuracy (Grad Requirement)

In [ ]:
N_SAMPLES = 3000

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

indices    = np.random.choice(len(val_ds), N_SAMPLES, replace=False)
imgs_clean = torch.stack([val_ds[i][0] for i in indices])
labels     = torch.tensor([val_ds[i][1] for i in indices])

noise_fns = {
    "Gaussian  (σ=0.10)"     : AddGaussianNoise(lo=0.10, hi=0.10),
    "Salt&Pepper (d=0.05)"   : AddSaltAndPepperNoise(lo=0.05, hi=0.05),
}

print(f"Per-noise accuracy on {N_SAMPLES} validation samples:\n")
accs = {}
with torch.no_grad():
    for name, fn in noise_fns.items():
        noisy  = torch.stack([fn(img) for img in imgs_clean]).to(DEVICE)
        preds  = model(noisy).argmax(1)
        acc    = (preds == labels.to(DEVICE)).float().mean().item()
        accs[name] = acc
        status = "PASS" if acc >= 0.95 else "FAIL"
        print(f"  {name:30s}  acc={acc:.4f}  [{status}]")

# Visual comparison: clean vs noisy
fig, axes = plt.subplots(2, 10, figsize=(14, 4))
fig.suptitle("Top: clean  |  Bottom: Gaussian noise", fontsize=11)
for i in range(10):
    img = imgs_clean[i]
    axes[0, i].imshow(img.squeeze(), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(AddGaussianNoise(lo=0.10, hi=0.10)(img).squeeze(), cmap="gray"); axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 10. Save Confirmation

In [ ]:
print(f"Best model saved to : {SAVE_PATH}")
print(f"Best val accuracy   : {best_val_acc:.4f}")
size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f"File size           : {size_mb:.2f} MB")